In [ ]:
# Week6 — Deep Learning Model Development (BTS On-Time)
# One cell to run end-to-end (Jupyter-friendly). Outputs saved under ./bts_on_time_data/eda/week6_*.*

# =========================
# Config
# =========================
ROOT_DIR   = "./bts_on_time_data"
SAMPLE_REL = "eda/sample_100k_week2_features.parquet"
SEED       = 42

# DL
N_EPOCHS   = 25
BATCH      = 1024
CALIB_BINS = 15
FP_COST    = 1.0
FN_COST    = 5.0
GRID_HIDDENS = [(128,64), (256,128), (96,48)]
GRID_DROPS   = [0.0, 0.15]
GRID_LRS     = [1e-3, 3e-4]

# =========================
# Imports
# =========================
import os, json, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, brier_score_loss
)

warnings.filterwarnings("ignore")

# =========================
# Paths
# =========================
ROOT = Path(ROOT_DIR)
EDA  = ROOT/"eda"
EDA.mkdir(parents=True, exist_ok=True)
SAMPLE = ROOT/SAMPLE_REL

# =========================
# Utilities
# =========================
def set_seed(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
set_seed(SEED)

def ensure_datetime(df):
    if "FlightDate" in df.columns and not np.issubdtype(df["FlightDate"].dtype, np.datetime64):
        df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

def safe_carrier_col(df):
    for c in ["Reporting_Airline","IATA_CODE_Reporting_Airline","UniqueCarrier"]:
        if c in df.columns: return c
    raise KeyError("Carrier column not found.")

def _on_rate(s: pd.Series) -> float:
    x = pd.to_numeric(s, errors="coerce").astype(float)
    x = x[np.isfinite(x)]
    return float(1.0 - x.mean()) if len(x) else np.nan

def _hour_bin(x):
    try: return int((float(x)//60)%24)
    except Exception: return np.nan

def _daily_group(df, group_cols):
    g = (df.groupby(group_cols + ["FlightDate"], dropna=False)
           .agg(n=("ArrDel15","count"),
                ontime=("ArrDel15", _on_rate),
                med_arr=("ArrDelay","median"))
           .reset_index())
    return g.sort_values("FlightDate")

def _rolling_join(hist, target, group_cols, windows=(7,14,30), prefix="route_"):
    dg = _daily_group(hist, group_cols).set_index("FlightDate")
    out = target.copy()
    for w in windows:
        on_w = (dg.groupby(group_cols)["ontime"]
                  .rolling(window=f"{w}D", min_periods=1).mean()
                  .shift(1).reset_index()
                  .rename(columns={"ontime": f"{prefix}on_{w}d"}))
        med30 = (dg.groupby(group_cols)["med_arr"]
                   .rolling(window="30D", min_periods=1).median()
                   .shift(1).reset_index()
                   .rename(columns={"med_arr": f"{prefix}med_30d"}))
        merged = on_w.merge(med30, on=group_cols+["FlightDate"], how="outer")
        m = out[group_cols + ["FlightDate"]].merge(merged, on=group_cols+["FlightDate"], how="left")
        out[f"{prefix}on_{w}d"]  = m.get(f"{prefix}on_{w}d")
        out[f"{prefix}med_30d"] = m.get(f"{prefix}med_30d")
    return out

def _hourly_load(hist, target, col_place, prefix):
    dfh = hist.copy()
    dfh["hour"] = dfh["CRSDepTime_min"].apply(_hour_bin)
    grp = (dfh.groupby([col_place,"hour","FlightDate"], dropna=False)
              .size().reset_index(name="dep")).set_index("FlightDate")
    out = target.copy()
    out["hour"] = out["CRSDepTime_min"].apply(_hour_bin)
    roll = (grp.groupby([col_place,"hour"])["dep"]
              .rolling(window="7D", min_periods=1).mean()
              .shift(1).reset_index()
              .rename(columns={"dep": f"{prefix}_hour_load_7d"}))
    m = out[[col_place,"hour","FlightDate"]].merge(roll, on=[col_place,"hour","FlightDate"], how="left")
    out[f"{prefix}_hour_load_7d"] = m.get(f"{prefix}_hour_load_7d")
    return out.drop(columns=["hour"])

def build_profiles_train(df_train, carrier_col):
    r = (df_train.groupby(["Origin","Dest"], dropna=False)
         .agg(route_med_arr_delay=("ArrDelay", "median"),
              route_ontime_rate=("ArrDel15", _on_rate),
              route_flights=("ArrDel15", "count"))
         .reset_index())
    c = (df_train.groupby([carrier_col], dropna=False)
         .agg(carrier_med_arr_delay=("ArrDelay", "median"),
              carrier_ontime_rate=("ArrDel15", _on_rate),
              carrier_flights=("ArrDel15", "count"))
         .reset_index())
    return r, c

def attach_profiles(df, rprof, cprof, carrier_col):
    df2 = df.merge(rprof, on=["Origin","Dest"], how="left") \
            .merge(cprof, on=[carrier_col],    how="left")
    return df2

def build_timeaware_features(hist_df, target_df, carrier_col):
    out = _rolling_join(hist_df, target_df, ["Origin","Dest"], windows=(7,14,30), prefix="route_")
    out = _rolling_join(hist_df, out, [carrier_col], windows=(7,14,30), prefix="car_")
    out = _hourly_load(hist_df, out, "Origin", "orig")
    out = _hourly_load(hist_df, out, "Dest", "dest")
    return out

def split_temporal(df):
    train = df[df["Year"] == 2024].copy()
    val   = df[(df["Year"] == 2025) & (df["Month"].isin([1,2,3]))].copy()
    test  = df[(df["Year"] == 2025) & (df["Month"] == 4)].copy()
    return train, val, test

def fillna_median(df, cols):
    for c in cols:
        if c in df.columns:
            med = pd.to_numeric(df[c], errors="coerce").median()
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(med)
    return df

def choose_best_threshold_f1(y_true, y_score):
    p, r, t = precision_recall_curve(y_true, y_score)
    f1 = 2*(p*r)/(p+r+1e-12)
    idx = int(np.nanargmax(f1))
    thr = float(t[idx-1]) if idx>0 else 0.5
    return thr, float(f1[idx]), float(p[idx]), float(r[idx])

def confusion_at(y_true, y_prob, thr):
    pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"threshold": float(thr), "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

def business_cost_at(y_true, y_prob, thr, fp_cost=1.0, fn_cost=5.0):
    cm = confusion_at(y_true, y_prob, thr)
    return fp_cost*cm["FP"] + fn_cost*cm["FN"], cm

def save_pr_curve(y_true, y_prob, out_png, title):
    pr, rc, _ = precision_recall_curve(y_true, y_prob)
    plt.figure()
    plt.plot(rc, pr); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=130); plt.close()

def save_reliability_plot(y_true, y_prob, out_png, bins=CALIB_BINS, title="Reliability"):
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=bins, strategy="quantile")
    plt.figure()
    plt.plot([0,1], [0,1], "--")
    plt.plot(prob_pred, prob_true, "o-")
    plt.xlabel("Predicted"); plt.ylabel("Observed"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=130); plt.close()

def temperature_scale_fit(y_true, p_val):
    eps = 1e-6
    y = np.asarray(y_true).astype(int)
    p = np.clip(np.asarray(p_val), eps, 1-eps)
    z = np.log(p/(1-p))
    T = keras.Variable(1.0, dtype=tf.float32)
    opt = keras.optimizers.Adam(learning_rate=0.05)
    for _ in range(200):
        with tf.GradientTape() as tape:
            zT = z / (T + 1e-6)
            pT = 1.0/(1.0+tf.exp(-zT))
            nll = -tf.reduce_mean(y*tf.math.log(pT+eps) + (1-y)*tf.math.log(1.0-pT+eps))
        g = tape.gradient(nll, [T])
        opt.apply_gradients(zip(g, [T]))
    return float(T.numpy())

def apply_temperature(p, T):
    eps = 1e-6
    p = np.clip(p, eps, 1-eps)
    z = np.log(p/(1-p))
    zT = z / max(T, eps)
    pT = 1.0/(1.0+np.exp(-zT))
    return pT

# =========================
# Load & Features
# =========================
print("[LOAD]", SAMPLE)
assert SAMPLE.exists(), f"Missing: {SAMPLE}"
df = pd.read_parquet(SAMPLE)
df = ensure_datetime(df)
carrier_col = safe_carrier_col(df)

train0, val0, test0 = split_temporal(df)
rprof, cprof = build_profiles_train(train0, carrier_col)

train0 = attach_profiles(train0, rprof, cprof, carrier_col)
val0   = attach_profiles(val0,   rprof, cprof, carrier_col)
test0  = attach_profiles(test0,  rprof, cprof, carrier_col)

train = build_timeaware_features(train0, train0, carrier_col)
val   = build_timeaware_features(train0, val0,   carrier_col)
test  = build_timeaware_features(pd.concat([train0,val0], ignore_index=True), test0, carrier_col)

for s in (train,val,test):
    s["ArrDel15"] = pd.to_numeric(s["ArrDel15"], errors="coerce").fillna(0).astype(int)

CAT_COLS = [carrier_col, "Origin","Dest","tod_bin","season"]
NUM_COLS_CORE = ["Distance","CRSDepTime_min","dow","is_weekend","quarter","DepDelay","TaxiOut"]
NUM_COLS_ROLL = [c for c in train.columns if any(c.startswith(p) for p in ["route_on_","route_med_","car_on_","car_med_","orig_hour_load_","dest_hour_load_"])]
NUM_COLS_PROFILE = ["route_med_arr_delay","route_ontime_rate","route_flights",
                    "carrier_med_arr_delay","carrier_ontime_rate","carrier_flights"]
NUM_COLS = [c for c in NUM_COLS_CORE + NUM_COLS_ROLL + NUM_COLS_PROFILE if c in train.columns]

for s in (train,val,test): fillna_median(s, NUM_COLS)
y_tr, y_va, y_te = train["ArrDel15"].values, val["ArrDel15"].values, test["ArrDel15"].values

# =========================
# Keras tabular model (embeddings + numerics)
# =========================
def make_vocabs(train_df, cols):
    voc = {}
    for c in cols:
        vc = (train_df[c].astype(str).fillna("NA").value_counts().index.tolist())
        voc[c] = ["<UNK>"] + vc
    return voc

VOCABS = make_vocabs(train, CAT_COLS)

def build_dataset(df, num_cols, cat_cols, vocabs):
    X = {}
    for c in num_cols: X[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("float32").values
    for c in cat_cols:
        s = df[c].astype(str).fillna("NA")
        mp = {v:i for i,v in enumerate(vocabs[c])}
        X[c] = s.map(mp).fillna(0).astype("int32").values
    return X

Xtr = build_dataset(train, NUM_COLS, CAT_COLS, VOCABS)
Xva = build_dataset(val,   NUM_COLS, CAT_COLS, VOCABS)
Xte = build_dataset(test,  NUM_COLS, CAT_COLS, VOCABS)

def tabular_model(hidden=(128,64), drop=0.0, lr=1e-3):
    num_ins = [keras.Input(shape=(1,), name=c, dtype="float32") for c in NUM_COLS]
    cat_ins = [keras.Input(shape=(1,), name=c, dtype="int32") for c in CAT_COLS]

    # numerics
    num_concat = keras.layers.Concatenate()(num_ins) if len(num_ins)>1 else num_ins[0]
    norm = keras.layers.Normalization()
    norm.adapt(np.stack([Xtr[c] for c in NUM_COLS], axis=1))
    x_num = norm(num_concat)

    # embeddings
    emb_list = []
    for i,c in enumerate(CAT_COLS):
        vocab_size = len(VOCABS[c])
        d = int(min(50, max(4, round(np.sqrt(vocab_size)))))
        e = keras.layers.Embedding(input_dim=vocab_size, output_dim=d, name=f"emb_{c}")(cat_ins[i])
        e = keras.layers.Reshape((d,))(e)
        emb_list.append(e)
    x_cat = keras.layers.Concatenate()(emb_list) if len(emb_list)>1 else emb_list[0]

    x = keras.layers.Concatenate()([x_num, x_cat])
    for h in hidden:
        x = keras.layers.Dense(h, activation="relu")(x)
        if drop>0: x = keras.layers.Dropout(drop)(x)
    out = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=num_ins+cat_ins, outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="ROC", curve="ROC"),
                           keras.metrics.AUC(name="PR",  curve="PR")])
    return model

def tf_inputs(Xdict):
    return {k: Xdict[k] for k in list(Xdict.keys())}

# Grid search (val PR-AUC)
print("[DL] tuning (small grid) ...")
best = None
grid = [(h,d,lr) for h in GRID_HIDDENS for d in GRID_DROPS for lr in GRID_LRS]
for (h,d,lr) in tqdm(grid, leave=False, desc="grid"):
    set_seed(SEED)
    mdl = tabular_model(hidden=h, drop=d, lr=lr)
    es = keras.callbacks.EarlyStopping(monitor="val_PR", mode="max", patience=3, restore_best_weights=True, verbose=0)
    hist = mdl.fit(tf_inputs(Xtr), y_tr, epochs=N_EPOCHS, batch_size=BATCH,
                   validation_data=(tf_inputs(Xva), y_va), callbacks=[es], verbose=0)
    pr_val = float(max(hist.history["val_PR"]))
    if (best is None) or (pr_val > best[0]):
        best = (pr_val, {"hidden":h, "drop":d, "lr":lr}, mdl, hist)

best_pr, best_cfg, best_model, best_hist = best
print(f"[DL] best cfg={best_cfg}  val PR-AUC={best_pr:.3f}")

# Train curves
plt.figure(); plt.plot(best_hist.history["ROC"]); plt.plot(best_hist.history["val_ROC"]); plt.legend(["train","val"])
plt.title("TF training ROC"); plt.tight_layout(); plt.savefig(EDA/"week6_tf_training_roc.png", dpi=130); plt.close()
plt.figure(); plt.plot(best_hist.history["PR"]); plt.plot(best_hist.history["val_PR"]); plt.legend(["train","val"])
plt.title("TF training PR"); plt.tight_layout(); plt.savefig(EDA/"week6_tf_training_pr.png", dpi=130); plt.close()

# Uncalibrated test probs
p_va_unc = best_model.predict(tf_inputs(Xva), batch_size=BATCH, verbose=0).ravel()
p_te_unc = best_model.predict(tf_inputs(Xte), batch_size=BATCH, verbose=0).ravel()

roc_unc = float(roc_auc_score(y_te, p_te_unc))
pr_unc  = float(average_precision_score(y_te, p_te_unc))
brier_unc = float(brier_score_loss(y_te, p_te_unc))
thr_f1, f1b, pb, rb = choose_best_threshold_f1(y_va, p_va_unc)
cm05_unc = confusion_at(y_te, p_te_unc, 0.5)
cmb_unc  = confusion_at(y_te, p_te_unc, thr_f1)

# Cost curve on val
ths = np.linspace(0.01, 0.99, 199)
costs = [business_cost_at(y_va, p_va_unc, t, FP_COST, FN_COST)[0] for t in ths]
thr_cost = float(ths[int(np.argmin(costs))])
pd.DataFrame({"thr": ths, "val_cost": costs}).to_csv(EDA/"week6_cost_curve_val.csv", index=False)

# Calibration (temperature on val)
T = temperature_scale_fit(y_va, p_va_unc)
p_te_cal = apply_temperature(p_te_unc, T)
roc_cal = float(roc_auc_score(y_te, p_te_cal))
pr_cal  = float(average_precision_score(y_te, p_te_cal))
brier_cal = float(brier_score_loss(y_te, p_te_cal))

# Save probs
pd.DataFrame({"y_true": y_te, "y_prob": p_te_unc}).to_csv(EDA/"week6_nn_prob_uncal.csv", index=False)
pd.DataFrame({"y_true": y_te, "y_prob": p_te_cal}).to_csv(EDA/"week6_nn_prob_cal.csv", index=False)

# PR & Reliability
save_pr_curve(y_te, p_te_unc, EDA/"week6_nn_pr_uncal.png", "PR — NN (uncal)")
save_pr_curve(y_te, p_te_cal, EDA/"week6_nn_pr_cal.png", "PR — NN (cal)")
save_reliability_plot(y_te, p_te_unc, EDA/"week6_nn_reliability_uncal.png", CALIB_BINS, "Reliability — NN (uncal)")
save_reliability_plot(y_te, p_te_cal, EDA/"week6_nn_reliability_cal.png", CALIB_BINS, "Reliability — NN (cal)")

# Thresholded confusion (val-derived best & cost-optimal)
cost_test_unc, cm_cost_unc = business_cost_at(y_te, p_te_unc, thr_cost, FP_COST, FN_COST)
cost_test_cal, cm_cost_cal = business_cost_at(y_te, p_te_cal, thr_cost, FP_COST, FN_COST)

# Permutation importance on numerics (drop in PR-AUC on TEST)
perm_rows = []
base_pr = average_precision_score(y_te, p_te_unc)
for c in tqdm(NUM_COLS, desc="[PERM] numerics", leave=False):
    Xtmp = dict(Xte)
    x = Xtmp[c].copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(x)
    Xtmp[c] = x
    p_tmp = best_model.predict(tf_inputs(Xtmp), batch_size=BATCH, verbose=0).ravel()
    pr_drop = float(base_pr - average_precision_score(y_te, p_tmp))
    perm_rows.append({"feature": c, "PR_drop": pr_drop})
pd.DataFrame(perm_rows).sort_values("PR_drop", ascending=False).to_csv(EDA/"week6_perm_importance_numeric.csv", index=False)

# Month-by-month backtest (DL)
months_pool = pd.concat([val.copy(), test.copy()], ignore_index=True)
def month_metrics(df_chunk):
    Xc = build_dataset(df_chunk, NUM_COLS, CAT_COLS, VOCABS)
    p  = best_model.predict(tf_inputs(Xc), batch_size=BATCH, verbose=0).ravel()
    return {"n": len(df_chunk),
            "ROC_AUC": float(roc_auc_score(df_chunk["ArrDel15"].values, p)),
            "PR_AUC": float(average_precision_score(df_chunk["ArrDel15"].values, p))}
rows = []
for (yy,mm), g in months_pool.groupby(["Year","Month"]):
    if len(g)>=200:
        m = month_metrics(g)
        m.update({"Year":int(yy), "Month":int(mm)})
        rows.append(m)
pd.DataFrame(rows).sort_values(["Year","Month"]).to_csv(EDA/"week6_backtest_dl.csv", index=False)

# Summary JSON
summary = {
    "config": {"epochs": N_EPOCHS, "batch": BATCH, "fp_cost": FP_COST, "fn_cost": FN_COST,
               "grid": {"hiddens": GRID_HIDDENS, "drops": GRID_DROPS, "lrs": GRID_LRS}},
    "best_cfg": best_cfg,
    "uncal": {"ROC_AUC": roc_unc, "PR_AUC": pr_unc, "Brier": brier_unc,
              "cm@0.5": cm05_unc, "cm@bestF1(val)": cmb_unc,
              "thr_bestF1_from_val": float(thr_f1),
              "thr_cost_from_val": thr_cost,
              "cost_test": float(cost_test_unc),
              "cost_cm": cm_cost_unc},
    "calibrated": {"T": T, "ROC_AUC": roc_cal, "PR_AUC": pr_cal, "Brier": brier_cal,
                   "thr_cost_from_val": thr_cost,
                   "cost_test": float(cost_test_cal),
                   "cost_cm": cm_cost_cal},
    "files": {
        "probs_uncal": str(EDA/"week6_nn_prob_uncal.csv"),
        "probs_cal":   str(EDA/"week6_nn_prob_cal.csv"),
        "pr_uncal_png": str(EDA/"week6_nn_pr_uncal.png"),
        "pr_cal_png":   str(EDA/"week6_nn_pr_cal.png"),
        "rel_uncal_png": str(EDA/"week6_nn_reliability_uncal.png"),
        "rel_cal_png":   str(EDA/"week6_nn_reliability_cal.png"),
        "cost_curve_val_csv": str(EDA/"week6_cost_curve_val.csv"),
        "perm_numeric_csv": str(EDA/"week6_perm_importance_numeric.csv"),
        "backtest_csv": str(EDA/"week6_backtest_dl.csv"),
        "train_roc_png": str(EDA/"week6_tf_training_roc.png"),
        "train_pr_png":  str(EDA/"week6_tf_training_pr.png")
    },
    "features": {"num": NUM_COLS, "cat": CAT_COLS, "y_clf": "ArrDel15"}
}
with open(EDA/"week6_metrics.json","w") as f:
    json.dump(summary, f, indent=2)

# Console digest
def f3(x): 
    try: return f"{float(x):.3f}"
    except: return x

print("\n=== Week6 — NN (TEST) ===")
print(f"Uncalibrated :: ROC={f3(roc_unc)}  PR={f3(pr_unc)}  Brier={f3(brier_unc)}  bestF1_thr={f3(thr_f1)}")
print(f"Calibrated(T={f3(T)}) :: ROC={f3(roc_cal)}  PR={f3(pr_cal)}  Brier={f3(brier_cal)}  cost_thr={f3(thr_cost)}")
print(f"cm@0.5 (uncal) = {cm05_unc}")
print(f"cm@bestF1(val) (uncal) = {cmb_unc}")
print(f"[DONE] Summary JSON -> {EDA/'week6_metrics.json'}")
arts = sorted([p.name for p in EDA.glob('week6_*')])
print("\n=== Week6 Artifacts under ./bts_on_time_data/eda ===")
for a in arts: print(" -", a)

[LOAD] bts_on_time_data/eda/sample_100k_week2_features.parquet
[DL] tuning (small grid) ...


grid:   0%|          | 0/12 [00:00<?, ?it/s]

[DL] best cfg={'hidden': (128, 64), 'drop': 0.0, 'lr': 0.001}  val PR-AUC=0.911


[PERM] numerics:   0%|          | 0/19 [00:00<?, ?it/s]


=== Week6 — NN (TEST) ===
Uncalibrated :: ROC=0.956  PR=0.911  Brier=0.064  bestF1_thr=0.110
Calibrated(T=1.452) :: ROC=0.956  PR=0.911  Brier=0.060  cost_thr=0.050
cm@0.5 (uncal) = {'threshold': 0.5, 'TP': 959, 'FP': 13, 'TN': 6707, 'FN': 673}
cm@bestF1(val) (uncal) = {'threshold': 0.10974982380867004, 'TP': 1298, 'FP': 170, 'TN': 6550, 'FN': 334}
[DONE] Summary JSON -> bts_on_time_data/eda/week6_metrics.json

=== Week6 Artifacts under ./bts_on_time_data/eda ===
 - week6_backtest_dl.csv
 - week6_cost_curve_val.csv
 - week6_metrics.json
 - week6_nn_pr_cal.png
 - week6_nn_pr_uncal.png
 - week6_nn_prob_cal.csv
 - week6_nn_prob_uncal.csv
 - week6_nn_reliability_cal.png
 - week6_nn_reliability_uncal.png
 - week6_perm_importance_numeric.csv
 - week6_tf_training_pr.png
 - week6_tf_training_roc.png


In [3]:
# === Week6 v2 — Improved Deep Learning on BTS (fixed calibration & thresholds) ===
# All outputs -> ./bts_on_time_data/eda/ with prefix: week6_v2_*
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, confusion_matrix
)
from sklearn.isotonic import IsotonicRegression

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --------------------
# Config
# --------------------
ROOT   = Path("./bts_on_time_data")
EDA    = ROOT/"eda"; EDA.mkdir(parents=True, exist_ok=True)
SAMPLE = EDA/"sample_100k_week2_features.parquet"
SEED   = 42

EPOCHS  = 50
BATCH   = 512
LR      = 1e-3
DROPOUT = 0.3
HIDDEN  = (128, 64, 32)

N_CAL_BINS = 15
FP_COST = 1.0
FN_COST = 5.0

DO_BACKTEST = True
PFX = "week6_v2"

# --------------------
# Utilities
# --------------------
REQ_BASE_COLS = ["Year","Month","DayofMonth","FlightDate","Origin","Dest",
                 "ArrDelay","ArrDel15","Distance","CRSDepTime_min","dow","is_weekend",
                 "quarter","season","tod_bin"]
PROFILE_COLS = [
    "route_med_arr_delay","route_ontime_rate","route_flights",
    "carrier_med_arr_delay","carrier_ontime_rate","carrier_flights"
]
ROLL_COLS = [
    "route_on_7d","route_on_14d","route_on_30d","route_med_30d",
    "car_on_7d","car_on_14d","car_on_30d","car_med_30d",
    "orig_hour_load_7d","dest_hour_load_7d"
]

def set_seed(sd=SEED):
    np.random.seed(sd); tf.random.set_seed(sd)

def ensure_datetime(df):
    if "FlightDate" in df.columns and not np.issubdtype(df["FlightDate"].dtype, np.datetime64):
        df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

def safe_carrier_col(df):
    for c in ["Reporting_Airline","IATA_CODE_Reporting_Airline","UniqueCarrier"]:
        if c in df.columns: return c
    raise KeyError("Carrier column not found.")

def _on_rate(s: pd.Series) -> float:
    x = pd.to_numeric(s, errors="coerce").astype(float).values
    x = x[~np.isnan(x)]
    if x.size == 0: return np.nan
    return 1.0 - float(np.mean(x))

def _daily_group(df, group_cols):
    g = (df.groupby(group_cols + ["FlightDate"], dropna=False)
           .agg(n=("ArrDel15","count"),
                ontime=("ArrDel15", _on_rate),
                med_arr=("ArrDelay","median"))
           .reset_index())
    return g.sort_values("FlightDate")

def _hour_bin(x):
    try: return int((float(x)//60)%24)
    except: return np.nan

def _rolling_join(hist, target, group_cols, windows=(7,14,30), prefix="route_"):
    dg = _daily_group(hist, group_cols).set_index("FlightDate")
    out = target.copy()
    for w in windows:
        on_w = (dg.groupby(group_cols)["ontime"]
                  .rolling(window=f"{w}D", min_periods=1).mean()
                  .shift(1).reset_index()
                  .rename(columns={"ontime": f"{prefix}on_{w}d"}))
        med30 = (dg.groupby(group_cols)["med_arr"]
                   .rolling(window="30D", min_periods=1).median()
                   .shift(1).reset_index()
                   .rename(columns={"med_arr": f"{prefix}med_30d"}))
        merged = on_w.merge(med30, on=group_cols+["FlightDate"], how="outer")
        m = out[group_cols + ["FlightDate"]].merge(merged, on=group_cols+["FlightDate"], how="left")
        out[f"{prefix}on_{w}d"]  = m.get(f"{prefix}on_{w}d")
        out[f"{prefix}med_30d"] = m.get(f"{prefix}med_30d")
    return out

def _hourly_load(hist, target, col_place, prefix):
    dfh = hist.copy()
    dfh["hour"] = dfh["CRSDepTime_min"].apply(_hour_bin)
    grp = (dfh.groupby([col_place,"hour","FlightDate"], dropna=False)
              .size().reset_index(name="dep")).set_index("FlightDate")
    out = target.copy()
    out["hour"] = out["CRSDepTime_min"].apply(_hour_bin)
    roll = (grp.groupby([col_place,"hour"])["dep"]
              .rolling(window="7D", min_periods=1).mean()
              .shift(1).reset_index()
              .rename(columns={"dep": f"{prefix}_hour_load_7d"}))
    m = out[[col_place,"hour","FlightDate"]].merge(roll, on=[col_place,"hour","FlightDate"], how="left")
    out[f"{prefix}_hour_load_7d"] = m.get(f"{prefix}_hour_load_7d")
    return out.drop(columns=["hour"])

def build_timeaware_features(hist_df, target_df, carrier_col):
    out = _rolling_join(hist_df, target_df, ["Origin","Dest"], windows=(7,14,30), prefix="route_")
    out = _rolling_join(hist_df, out, [carrier_col], windows=(7,14,30), prefix="car_")
    out = _hourly_load(hist_df, out, "Origin", "orig")
    out = _hourly_load(hist_df, out, "Dest", "dest")
    for c in ROLL_COLS:
        if c in out.columns: out[c] = out[c].fillna(out[c].median())
    return out

def split_temporal(df):
    train = df[df["Year"] == 2024].copy()
    val   = df[(df["Year"] == 2025) & (df["Month"].isin([1,2,3]))].copy()
    test  = df[(df["Year"] == 2025) & (df["Month"] == 4)].copy()
    return train, val, test

def setup_feature_spaces(df, carrier_col):
    cat = [carrier_col, "Origin", "Dest", "tod_bin", "season"]
    num_core = ["Distance", "CRSDepTime_min", "dow", "is_weekend", "quarter"]
    num = num_core + PROFILE_COLS + ROLL_COLS
    if "DepDelay" in df.columns: num.append("DepDelay")
    if "TaxiOut"  in df.columns: num.append("TaxiOut")
    y_clf = "ArrDel15"
    groups = {"CORE": num_core, "PROFILE": PROFILE_COLS, "ROLLING": ROLL_COLS}
    return num, cat, y_clf, groups

def build_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc", StandardScaler())])
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                         ("ohe", ohe)])
    return ColumnTransformer([("num", num_pipe, num_cols),
                              ("cat", cat_pipe, cat_cols)], remainder="drop")

def class_weights_auto(y):
    y = np.asarray(y).ravel().astype(int)
    p = (y==1).mean()
    p = p if p>0 else 1e-6
    return {0: float(0.5/(1-p)), 1: float(0.5/p)}

def pr_plot(y_true, y_prob, out_png, title):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    plt.figure(figsize=(6,4))
    plt.plot(r, p)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(title); plt.tight_layout(); plt.savefig(out_png, dpi=140); plt.close()

def reliability_plot(y_true, y_prob, out_png, title):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=N_CAL_BINS, strategy="quantile")
    plt.figure(figsize=(6,4))
    plt.plot([0,1],[0,1],"--",label="ideal")
    plt.plot(prob_pred, prob_true, "o-", label="observed")
    plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.savefig(out_png, dpi=140); plt.close()

def best_f1_threshold(y_true, y_prob):
    p, r, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2*p*r/(p+r+1e-12)
    idx = int(np.nanargmax(f1))
    if idx <= 0: return 0.5
    return float(thr[min(idx, len(thr)-1)])

def confusion_at(y_true, y_prob, thr):
    pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"threshold": float(thr), "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

def business_cost_at(y_true, y_prob, thr, fp_cost=FP_COST, fn_cost=FN_COST):
    cm = confusion_at(y_true, y_prob, thr)
    return fp_cost*cm["FP"] + fn_cost*cm["FN"], cm

def temp_scale_fit(p_val, y_val, max_steps=200, lr=0.1):
    p = np.clip(p_val, 1e-6, 1-1e-6)
    logit = np.log(p/(1-p)).astype(np.float32)
    ytf   = tf.constant(y_val.astype(np.float32))
    logtf = tf.constant(logit)
    T = tf.Variable(1.0, dtype=tf.float32)
    opt = keras.optimizers.Adam(lr)
    for _ in range(max_steps):
        with tf.GradientTape() as tape:
            pcal = tf.sigmoid(logtf / T)
            loss = tf.reduce_mean(keras.losses.binary_crossentropy(ytf, pcal))
        g = tape.gradient(loss, [T]); opt.apply_gradients(zip(g, [T]))
    return float(T.numpy())

def apply_T(p, T):
    p = np.clip(p, 1e-6, 1-1e-6)
    logit = np.log(p/(1-p))
    return 1/(1+np.exp(-logit/T))

# --------------------
# Load & features
# --------------------
set_seed(SEED)
assert SAMPLE.exists(), f"Missing features parquet: {SAMPLE}"
df = pd.read_parquet(SAMPLE)
df = ensure_datetime(df)
missing = set(REQ_BASE_COLS) - set(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
carrier_col = safe_carrier_col(df)

print("[SPLIT] temporal train(2024) / val(2025-01..03) / test(2025-04)")
train0, val0, test0 = split_temporal(df)
print("[FEATS] time-aware (rolling/leakage-safe)")
train = build_timeaware_features(train0, train0, carrier_col)
val   = build_timeaware_features(train0, val0,   carrier_col)
test  = build_timeaware_features(pd.concat([train0, val0], ignore_index=True), test0, carrier_col)

for s in (train, val, test):
    s["ArrDel15"] = pd.to_numeric(s["ArrDel15"], errors="coerce").fillna(0).astype(int)

num_cols, cat_cols, y_clf, GROUPS = setup_feature_spaces(train, carrier_col)
prep = build_preprocessor(num_cols, cat_cols)
print("[PREP] fitting ColumnTransformer ...")
prep.fit(train[num_cols + cat_cols])

def to_xy(s):
    X = prep.transform(s[num_cols + cat_cols])
    y = s[y_clf].values.astype(int)
    return X.astype(np.float32), y

X_train, y_train = to_xy(train)
X_val,   y_val   = to_xy(val)
X_test,  y_test  = to_xy(test)
print(f"[SHAPE] X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")

# --------------------
# Build & train MLP
# --------------------
def build_mlp(input_dim: int):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(HIDDEN[0], activation="relu",
                     kernel_initializer="he_normal", kernel_regularizer=keras.regularizers.l2(1e-5)),
        layers.BatchNormalization(),
        layers.Dropout(DROPOUT),
        layers.Dense(HIDDEN[1], activation="relu",
                     kernel_initializer="he_normal", kernel_regularizer=keras.regularizers.l2(1e-5)),
        layers.BatchNormalization(),
        layers.Dropout(DROPOUT),
        layers.Dense(HIDDEN[2], activation="relu",
                     kernel_initializer="he_normal", kernel_regularizer=keras.regularizers.l2(1e-5)),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="rocauc"),
                           keras.metrics.AUC(name="prauc", curve="PR")])
    return model

print("[DL] training improved MLP ...")
class_weight = class_weights_auto(y_train)
cb = [
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_prauc", mode="max"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-5, monitor="val_prauc", mode="max")
]
model = build_mlp(X_train.shape[1])
hist = model.fit(X_train, y_train,
                 validation_data=(X_val, y_val),
                 epochs=EPOCHS, batch_size=BATCH,
                 class_weight=class_weight, verbose=0, callbacks=cb)

plt.figure(figsize=(7,4)); plt.plot(hist.history["rocauc"], label="train"); plt.plot(hist.history["val_rocauc"], label="val")
plt.title("TF training ROC"); plt.legend(); plt.tight_layout(); plt.savefig(EDA/f"{PFX}_training_roc.png", dpi=140); plt.close()
plt.figure(figsize=(7,4)); plt.plot(hist.history["prauc"], label="train"); plt.plot(hist.history["val_prauc"], label="val")
plt.title("TF training PR"); plt.legend(); plt.tight_layout(); plt.savefig(EDA/f"{PFX}_training_pr.png", dpi=140); plt.close()

# --------------------
# Predict (val/test)
# --------------------
p_val  = model.predict(X_val,  verbose=0).ravel()
p_test = model.predict(X_test, verbose=0).ravel()

# --------------------
# Calibration: Isotonic & Temperature (fit on VAL, apply to VAL & TEST)
# --------------------
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_val, y_val)
p_val_iso  = iso.predict(p_val)
p_test_iso = iso.predict(p_test)

def temperature_scale_fit_and_apply(p_val, y_val, p_list):
    T = temp_scale_fit(p_val, y_val, max_steps=200, lr=0.1)
    scaled = [apply_T(p, T) for p in p_list]
    return T, scaled

Tval, [p_val_T, p_test_T] = temperature_scale_fit_and_apply(p_val, y_val, [p_val, p_test])

# --------------------
# Evaluation blocks (thresholds chosen on VAL of the SAME calibration)
# --------------------
def eval_block(tag, p_test_proba, p_val_for_thr):
    roc = roc_auc_score(y_test, p_test_proba)
    pr  = average_precision_score(y_test, p_test_proba)
    br  = brier_score_loss(y_test, p_test_proba)
    thr_b = best_f1_threshold(y_val, p_val_for_thr)   # <-- now consistent
    cm05  = confusion_at(y_test, p_test_proba, 0.5)
    cmb   = confusion_at(y_test, p_test_proba, thr_b)
    pr_plot(y_test, p_test_proba, EDA/f"{PFX}_pr_{tag}.png", f"PR — NN ({tag})")
    reliability_plot(y_test, p_test_proba, EDA/f"{PFX}_reliability_{tag}.png", f"Reliability — NN ({tag})")
    return {"ROC_AUC": float(roc), "PR_AUC": float(pr), "Brier": float(br),
            "cm@0.5": cm05, "cm@best_val": cmb, "best_thr_from_val": float(thr_b)}

res_uncal = eval_block("uncal", p_test,    p_val)
res_iso   = eval_block("iso",   p_test_iso, p_val_iso)
res_T     = eval_block("T",     p_test_T,   p_val_T)

# Business cost curve on VAL (uncal) -> apply to TEST (uncal)
ths = np.linspace(0.01, 0.99, 199)
val_costs = [business_cost_at(y_val, p_val, t, FP_COST, FN_COST)[0] for t in ths]
best_cost_thr = float(ths[int(np.argmin(val_costs))])
cost_test, cm_cost = business_cost_at(y_test, p_test, best_cost_thr, FP_COST, FN_COST)
pd.DataFrame({"thr": ths, "val_cost": val_costs}).to_csv(EDA/f"{PFX}_cost_curve_val.csv", index=False)

# Save probabilities
pd.DataFrame({
    "y_true": y_test, "y_prob_uncal": p_test,
    "y_prob_iso": p_test_iso, "y_prob_T": p_test_T
}).to_csv(EDA/f"{PFX}_prob_test.csv", index=False)

# --------------------
# Backtest (optional)
# --------------------
bt_csv = None
if DO_BACKTEST:
    print("[BACKTEST] month-by-month on 2025-01..04 (+ late-2024 if present)")
    months_pool = pd.concat([val.copy(), test.copy()], ignore_index=True)
    late_2024 = df[(df["Year"]==2024) & (df["Month"].isin([9,10,11,12]))].copy()
    if not late_2024.empty:
        late_2024 = build_timeaware_features(train0, late_2024, carrier_col)
        months_pool = pd.concat([late_2024, months_pool], ignore_index=True)
    rows = []
    for (yy, mm), chunk in months_pool.groupby(["Year","Month"]):
        Xm, ym = prep.transform(chunk[num_cols + cat_cols]).astype(np.float32), chunk[y_clf].values.astype(int)
        pm = model.predict(Xm, verbose=0).ravel()
        rows.append({"Year": int(yy), "Month": int(mm), "n": len(chunk),
                     "ROC_AUC": float(roc_auc_score(ym, pm)),
                     "PR_AUC": float(average_precision_score(ym, pm)),
                     "Brier": float(brier_score_loss(ym, pm))})
    bt = pd.DataFrame(rows).sort_values(["Year","Month"])
    bt_csv = EDA/f"{PFX}_backtest_dl.csv"
    bt.to_csv(bt_csv, index=False)

# --------------------
# Compare with Week5 CV (if available)
# --------------------
cmp_rows = []
week5_csv = EDA/"week5_cv_classification.csv"
if week5_csv.exists():
    w5 = pd.read_csv(week5_csv)
    agg = w5.groupby("model", as_index=False)[["ROC_AUC","PR_AUC","brier"]].mean()
    best = agg.sort_values("PR_AUC", ascending=False).head(1).iloc[0]
    cmp_rows.append({"source":"Week5_best_cv", "model":best["model"],
                     "ROC_AUC": float(best["ROC_AUC"]), "PR_AUC": float(best["PR_AUC"]),
                     "Brier": float(best.get("brier", np.nan))})
    cmp_rows += [
        {"source":"Week6_v2_uncal","model":"nn",           "ROC_AUC":res_uncal["ROC_AUC"],"PR_AUC":res_uncal["PR_AUC"],"Brier":res_uncal["Brier"]},
        {"source":"Week6_v2_iso",  "model":"nn+isotonic",  "ROC_AUC":res_iso["ROC_AUC"],  "PR_AUC":res_iso["PR_AUC"],  "Brier":res_iso["Brier"]},
        {"source":"Week6_v2_T",    "model":"nn+Tscale",    "ROC_AUC":res_T["ROC_AUC"],    "PR_AUC":res_T["PR_AUC"],    "Brier":res_T["Brier"]},
    ]
    pd.DataFrame(cmp_rows).to_csv(EDA/f"{PFX}_vs_week5.csv", index=False)

# --------------------
# Save JSON & Console
# --------------------
summary = {
    "config": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "dropout": DROPOUT, "hidden": HIDDEN,
               "fp_cost": FP_COST, "fn_cost": FN_COST, "n_cal_bins": N_CAL_BINS},
    "results": {
        "uncal": res_uncal,
        "isotonic": res_iso,
        "Tscale": {"T": float(Tval), **res_T},
        "thr_cost(val)": {"thr": best_cost_thr, "test_cost": float(cost_test), "cm@thr_cost(test)": cm_cost}
    },
    "artifacts": {
        "probs_csv": str(EDA/f"{PFX}_prob_test.csv"),
        "pr_uncal_png": str(EDA/f"{PFX}_pr_uncal.png"),
        "pr_iso_png":   str(EDA/f"{PFX}_pr_iso.png"),
        "pr_T_png":     str(EDA/f"{PFX}_pr_T.png"),
        "rel_uncal_png":str(EDA/f"{PFX}_reliability_uncal.png"),
        "rel_iso_png":  str(EDA/f"{PFX}_reliability_iso.png"),
        "rel_T_png":    str(EDA/f"{PFX}_reliability_T.png"),
        "cost_curve_val_csv": str(EDA/f"{PFX}_cost_curve_val.csv"),
        "training_roc_png": str(EDA/f"{PFX}_training_roc.png"),
        "training_pr_png":  str(EDA/f"{PFX}_training_pr.png"),
        "backtest_csv": str(bt_csv) if bt_csv else None,
        "vs_week5_csv": str(EDA/f"{PFX}_vs_week5.csv") if (EDA/f"{PFX}_vs_week5.csv").exists() else None
    }
}
with open(EDA/f"{PFX}_metrics.json","w") as f: json.dump(summary, f, indent=2)

def _f(x): 
    try: return f"{float(x):.3f}"
    except: return x

print("\n=== Week6 v2 — NN (TEST) ===")
print(f"Uncalibrated :: ROC={_f(res_uncal['ROC_AUC'])}  PR={_f(res_uncal['PR_AUC'])}  Brier={_f(res_uncal['Brier'])}  bestF1_thr={_f(res_uncal['best_thr_from_val'])}")
print(f"Isotonic     :: ROC={_f(res_iso['ROC_AUC'])}  PR={_f(res_iso['PR_AUC'])}  Brier={_f(res_iso['Brier'])}  bestF1_thr={_f(res_iso['best_thr_from_val'])}")
print(f"T-Scaling(T={_f(Tval)}) :: ROC={_f(res_T['ROC_AUC'])}  PR={_f(res_T['PR_AUC'])}  Brier={_f(res_T['Brier'])}  bestF1_thr={_f(res_T['best_thr_from_val'])}")
print(f"cm@0.5 (uncal) = {res_uncal['cm@0.5']}  cm@best(val, uncal) = {res_uncal['cm@best_val']}")
print(f"[COST] best thr on val={_f(best_cost_thr)}  test cost={_f(cost_test)}  cm@thr={cm_cost}")
if (EDA/f"{PFX}_vs_week5.csv").exists():
    cmp_df = pd.read_csv(EDA/f"{PFX}_vs_week5.csv")
    print("\n=== Week6 v2 vs Week5 (best CV model) ===")
    print(cmp_df.to_string(index=False))
print(f"\n[DONE] Summary JSON -> {EDA/(PFX + '_metrics.json')}")
print("Artifacts saved under:", EDA.resolve())

[SPLIT] temporal train(2024) / val(2025-01..03) / test(2025-04)
[FEATS] time-aware (rolling/leakage-safe)
[PREP] fitting ColumnTransformer ...
[SHAPE] X_train=(67485, 725)  X_val=(24163, 725)  X_test=(8352, 725)
[DL] training improved MLP ...
[BACKTEST] month-by-month on 2025-01..04 (+ late-2024 if present)

=== Week6 v2 — NN (TEST) ===
Uncalibrated :: ROC=0.963  PR=0.923  Brier=0.055  bestF1_thr=0.816
Isotonic     :: ROC=0.963  PR=0.918  Brier=0.043  bestF1_thr=0.479
T-Scaling(T=0.929) :: ROC=0.963  PR=0.923  Brier=0.055  bestF1_thr=0.832
cm@0.5 (uncal) = {'threshold': 0.5, 'TP': 1426, 'FP': 403, 'TN': 6317, 'FN': 206}  cm@best(val, uncal) = {'threshold': 0.8157827854156494, 'TP': 1305, 'FP': 124, 'TN': 6596, 'FN': 327}
[COST] best thr on val=0.465  test cost=1440.000  cm@thr={'threshold': 0.46535353535353535, 'TP': 1433, 'FP': 445, 'TN': 6275, 'FN': 199}

=== Week6 v2 vs Week5 (best CV model) ===
        source       model  ROC_AUC   PR_AUC    Brier
 Week5_best_cv      logreg 0.96207